# Laboratorio de Sistemas Operativos — Práctica No. 4
## API de Hilos (Thread API)

| | |
|---|---|
| **Facultad** | Ingeniería — Ingeniería de Sistemas |
| **Asignatura** | Laboratorio de Sistemas Operativos |
| **CPU** | AMD Ryzen 7 5825U — 16 núcleos lógicos |
| **SO** | Ubuntu (WSL2) |

---

# Sección 1 — Análisis de π

## Contexto

El valor de π se aproxima mediante la integral definida:

$$\int_0^1 \frac{4}{1+x^2}\,dx = \pi$$

El algoritmo divide el intervalo `[0,1]` en **n rectángulos** (método del punto medio) y suma sus áreas:

$$\pi \approx \frac{1}{n} \sum_{i=0}^{n-1} f\!\left(\frac{i+0.5}{n}\right), \quad f(x) = \frac{4}{1+x^2}$$

Se usó **n = 2 000 000 000** en todos los experimentos.

In [ ]:
# ============================================================
# Datos medidos en el sistema local (Ubuntu WSL2)
# CPU: AMD Ryzen 7 5825U — 16 núcleos lógicos
# n = 2_000_000_000
# ============================================================

Ts = 3.7935  # tiempo serial (pi_s)

# Tiempos paralelos medidos con ./pi_p 2000000000 T
mediciones = [
    {"T": 1,  "Tp": 3.7736},
    {"T": 2,  "Tp": 1.5957},
    {"T": 4,  "Tp": 1.0075},
    {"T": 8,  "Tp": 0.5595},
    {"T": 16, "Tp": 0.4673},
    {"T": 32, "Tp": 0.4833},
]

print(f"Ts (serial) = {Ts:.4f} s")
print(f"CPU: AMD Ryzen 7 5825U — 16 núcleos lógicos")

## 1.1 Tabla de Métricas de Rendimiento

In [ ]:
import pandas as pd

for m in mediciones:
    m["Speedup"]    = round(Ts / m["Tp"], 4)
    m["Eficiencia"] = round(m["Speedup"] / m["T"], 4)

df = pd.DataFrame(mediciones)[["T", "Tp", "Speedup", "Eficiencia"]]
df.columns = ["N Hilos", "Tp (segundos)", "Speedup (Ts/Tp)", "Eficiencia (Speedup/N)"]

# Fila del tiempo serial como referencia
fila_serial = pd.DataFrame([{
    "N Hilos": "Serial",
    "Tp (segundos)": Ts,
    "Speedup (Ts/Tp)": 1.0,
    "Eficiencia (Speedup/N)": 1.0
}])

df_completo = pd.concat([fila_serial, df], ignore_index=True)

print("Tabla 1 — Métricas de rendimiento para el cálculo paralelo de π")
print(f"Ts = {Ts:.4f} s | n = 2 000 000 000 | CPU: AMD Ryzen 7 5825U (16 núcleos)")
print("=" * 70)
display(df_completo)

## 1.2 Gráfico de Speedup y Eficiencia

In [ ]:
import matplotlib.pyplot as plt

T_vals  = [m["T"]          for m in mediciones]
sp_vals = [m["Speedup"]    for m in mediciones]
ef_vals = [m["Eficiencia"] for m in mediciones]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Métricas de Rendimiento — Cálculo Paralelo de π\n"
             "AMD Ryzen 7 5825U | n = 2 000 000 000",
             fontsize=13, fontweight="bold")

# ── Speedup ──────────────────────────────────────────────
ax1.plot(T_vals, sp_vals, "o-", color="#2563eb", linewidth=2.5,
         markersize=8, label="Speedup real")
ax1.plot(T_vals, T_vals,  "--", color="#94a3b8", linewidth=1.5,
         label="Speedup ideal (lineal)")
ax1.fill_between(T_vals, sp_vals, [min(s, t) for s, t in zip(sp_vals, T_vals)],
                 alpha=0.08, color="#2563eb")
ax1.set_xlabel("Número de Hilos (T)")
ax1.set_ylabel("Speedup (Ts / Tp)")
ax1.set_title("Speedup")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.set_xticks(T_vals)

# Anotar valores
for t, s in zip(T_vals, sp_vals):
    ax1.annotate(f"{s:.2f}", (t, s), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=9)

# ── Eficiencia ───────────────────────────────────────────
ax2.plot(T_vals, ef_vals, "s-", color="#16a34a", linewidth=2.5,
         markersize=8, label="Eficiencia real")
ax2.axhline(y=1.0, color="#94a3b8", linestyle="--",
            linewidth=1.5, label="Eficiencia ideal = 1")
ax2.set_xlabel("Número de Hilos (T)")
ax2.set_ylabel("Eficiencia (Speedup / T)")
ax2.set_title("Eficiencia")
ax2.set_ylim(0, 1.3)
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.set_xticks(T_vals)

for t, e in zip(T_vals, ef_vals):
    ax2.annotate(f"{e:.2f}", (t, e), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("speedup_pi.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado como speedup_pi.png")

## 1.3 Análisis de Resultados

### Comparación Tp(1) vs Ts

| Métrica | Valor |
|---|---|
| Ts (serial puro) | 3.7935 s |
| Tp(1) (paralelo, 1 hilo) | 3.7736 s |
| Diferencia | −0.0199 s |

Con **T = 1 hilo**, `Tp(1)` es marginalmente menor que `Ts` (≈ 0.5%). Esto puede deberse a variabilidad del sistema o al compilador optimizando la versión paralela de forma ligeramente diferente. En general se espera que `Tp(1) > Ts` por el *overhead* de crear un hilo, gestionar la struct de argumentos y realizar el `pthread_join`. Con n = 2 000 000 000, ese overhead (microsegundos) queda enmascarado por la duración de la carga de trabajo (~3.8 s).

### Speedup Máximo y Núcleos Físicos

El Speedup máximo observado es **S = 8.11 con T = 16 hilos**, sobre un procesador de 16 núcleos lógicos (8 núcleos físicos con SMT). El Speedup no alcanza el ideal lineal (S = 16) porque:

- **Ley de Amdahl**: la fracción serial del programa (crear hilos, agregar sumas parciales, `malloc`/`free`) limita el Speedup máximo teórico.
- **Overhead de Pthreads**: creación y destrucción de 16 hilos consume tiempo no paralelo.
- **Memoria compartida y caché**: con n = 2 000 000 000, todos los hilos ejecutan cálculo puro sin escritura compartida, por lo que la contención de caché es mínima. Esto explica el buen escalado hasta 16 hilos.

Con T = 32 (más hilos que núcleos), el Speedup cae levemente (S = 7.85) porque el SO debe hacer *time-slicing* entre hilos que compiten por los mismos núcleos.

### Tendencia de la Eficiencia

| T | Eficiencia |
|---|---|
| 1 | 1.006 |
| 2 | 1.188 |
| 4 | 0.940 |
| 8 | 0.847 |
| 16 | 0.507 |
| 32 | 0.245 |

La eficiencia **disminuye consistentemente** al aumentar T. Las causas son:
1. El overhead de Pthreads (creación, join) crece con T.
2. Al superar los 8 núcleos físicos, entran en juego núcleos lógicos SMT que comparten recursos (ALU, caché L1/L2), reduciendo la ganancia real por hilo adicional.
3. Con T = 32, muchos hilos compiten por los 16 núcleos lógicos, aumentando el costo de scheduling.

**Punto óptimo práctico:** T = 8 (Speedup = 6.78, Eficiencia = 0.85) ofrece el mejor balance entre rendimiento y uso eficiente de recursos.

---
# Sección 2 — Análisis de Fibonacci

## Contexto

La secuencia de Fibonacci se define como:

$$F(-2) = 0,\quad F(-1) = 1,\quad F(n) = F(n-1) + F(n-2) \quad (n \geq 0)$$

Produciendo: `0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, ...`

## 2.1 Resultados de Ejecución — `./fibonacci 15`

```
Secuencia de Fibonacci (N=15):
  F(0)  = 0
  F(1)  = 1
  F(2)  = 1
  F(3)  = 2
  F(4)  = 3
  F(5)  = 5
  F(6)  = 8
  F(7)  = 13
  F(8)  = 21
  F(9)  = 34
  F(10) = 55
  F(11) = 89
  F(12) = 144
  F(13) = 233
  F(14) = 377
```

## 2.2 Implementación Serial (sin hilos) y Prueba de Rendimiento

Para comparar, se implementa el cálculo de Fibonacci **sin hilos** y se prueba con N grande:

In [ ]:
import time

# Fibonacci serial (sin hilos) — equivalente al bucle en fibonacci.c
def fibonacci_serial(N):
    arr = [0] * N
    if N >= 1: arr[0] = 0
    if N >= 2: arr[1] = 1
    for i in range(2, N):
        arr[i] = arr[i-1] + arr[i-2]
    return arr

# Prueba de rendimiento para N > 100_000
for N in [100_000, 500_000, 1_000_000]:
    t0 = time.perf_counter()
    fibonacci_serial(N)
    t = time.perf_counter() - t0
    print(f"N = {N:>10,}  →  {t*1000:.2f} ms")

## 2.3 Análisis del Diseño

### Mecanismo de transferencia de datos

En `fibonacci.c` se utiliza una `struct FibArgs` que contiene:
- `long *arreglo` — puntero al bloque de memoria asignado por `main` con `malloc`.
- `int N` — cantidad de términos a generar.

Esta struct se pasa al hilo trabajador como `void *arg` en `pthread_create`. Dentro del hilo, el cast `(FibArgs *)arg` recupera ambos campos. El arreglo en sí **vive en el heap del proceso** y es visible tanto para `main` como para el hilo trabajador, porque los hilos de un mismo proceso comparten el espacio de direcciones (modelo de memoria compartida de Pthreads).

### Rol de `pthread_join` como sincronización

```
main:        malloc → pthread_create → pthread_join() ──[BLOQUEADO]──► imprime arreglo
trabajador:                            ──► calcula Fibonacci ──► pthread_exit ──► desbloquea main
```

`pthread_join` es **crítico** porque:

1. **Garantía de visibilidad:** sin el join, `main` podría leer el arreglo antes de que el trabajador haya terminado de llenarlo, obteniendo valores incorrectos (condición de carrera).
2. **Barrera de memoria:** el join establece una relación *happens-before*: todo lo que el trabajador escribió antes de `pthread_exit` es visible para `main` después del `pthread_join`.
3. **Sin join = comportamiento no determinista:** en sistemas con múltiples núcleos, `main` y el trabajador corren verdaderamente en paralelo; sin sincronización el resultado es impredecible.

### Por qué Fibonacci no escala con hilos

La secuencia de Fibonacci es **intrínsecamente serial**: cada término `F(n)` depende de `F(n-1)` y `F(n-2)`. No existe forma de dividir el cálculo en sub-rangos independientes como se hizo con π. El hilo trabajador en este ejercicio **no ofrece speedup**; su propósito es demostrar el patrón de memoria compartida y sincronización con `pthread_join`.

---
# Conclusiones

1. **Concurrencia ≠ Paralelismo:** crear hilos no garantiza ejecución simultánea. El verdadero paralelismo ocurre cuando los hilos se ejecutan en núcleos físicos distintos al mismo tiempo.

2. **El overhead de Pthreads es real pero amortizable:** con `n = 2 000 000 000`, el costo de crear/destruir hilos (~microsegundos) queda diluido en los ~3.8 s de cómputo. En problemas pequeños, ese overhead puede superar el beneficio del paralelismo.

3. **La Ley de Amdahl limita el Speedup máximo:** la fracción serial del programa (inicialización, suma de parciales) impone un techo teórico al Speedup independientemente de cuántos hilos se usen. El Speedup máximo observado (8.11×) es consistente con esta ley para una fracción serial pequeña pero no nula.

4. **La eficiencia decrece al aumentar T:** más hilos implica más overhead de sincronización y, cuando T supera los núcleos físicos disponibles, competencia por recursos de hardware (ALU, caché, scheduling), reduciendo la ganancia real por hilo.

5. **`pthread_join` es indispensable para coherencia:** actúa como barrera de sincronización que garantiza que todos los datos escritos por los hilos trabajadores sean visibles para `main` antes de que éste los lea. Sin él, se producen condiciones de carrera con resultados no deterministas.

6. **Data Parallelism es efectivo para cálculo numérico independiente:** dividir el rango `[0, n)` entre T hilos funciona porque cada rectángulo se calcula de forma completamente independiente. Esta independencia es lo que permite el alto Speedup observado.

7. **Fibonacci es intrínsecamente serial:** la dependencia secuencial `F(n) = F(n-1) + F(n-2)` impide paralelizar el cálculo. El ejercicio ilustra el patrón de memoria compartida y sincronización, no el paralelismo de datos.

8. **El punto óptimo es T = núcleos físicos:** con T = 8 (núcleos físicos del Ryzen 7 5825U) se obtiene Speedup = 6.78 y Eficiencia = 0.85, el mejor balance. Superar los núcleos físicos (usar hilos SMT) ofrece ganancia marginal decreciente.

---
# Referencias

- Arpaci-Dusseau, R. H. & Arpaci-Dusseau, A. C. *Operating Systems: Three Easy Pieces*.
  - Capítulo 26: Threads Intro
  - Capítulo 27: Thread API
- The Open Group. *IEEE Std 1003.1 — POSIX Threads (Pthreads) API*.
- Lawrence Livermore National Laboratory. *POSIX Threads Programming Tutorial*.